In [1]:
%pip install "qiskit==2.2.3" "qiskit-machine-learning==0.9.0" "qiskit-optimization==0.7.0" "qiskit-algorithms==0.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 20.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ec5f674bcf84d68d61b01cd3379caf1ddb984603e90ce9b06558131a8193dc9d
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [9]:
import sys
import importlib.metadata as metadata
import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_machine_learning
import qiskit_optimization
import qiskit_algorithms

expected = {
    "qiskit": "2.2.3",
    "qiskit-machine-learning": "0.9.0",
    "qiskit-optimization": "0.7.0",
    "qiskit-algorithms": "0.4.0",
}
print("Python:", sys.version.split()[0])
for package in [*expected, "numpy", "scipy", "scikit-learn"]:
    installed = metadata.version(package)
    print(f"{package}: {installed}")
    if package in expected:
        assert installed == expected[package], (
            f"{package} 버전을 확인하고 설치 셀 실행 후 세션을 다시 시작하세요."
        )
print("패키지 import와 버전 확인 완료")

Python: 3.13.18
qiskit: 2.2.3
qiskit-machine-learning: 0.9.0
qiskit-optimization: 0.7.0
qiskit-algorithms: 0.4.0
numpy: 2.4.6
scipy: 1.16.3
scikit-learn: 1.6.1
패키지 import와 버전 확인 완료


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.algorithms.classifiers import QSVC

normal = np.array([[.10,.12],[.15,.18],[.20,.16],[.25,.22],[.30,.28],[.35,.32],[.40,.36],[.45,.40],[.50,.44],[.55,.48],[.60,.52],[.65,.56]])
anomaly = np.array([[.35,.65],[.40,.70],[.45,.75],[.50,.80],[.55,.85],[.60,.90],[.65,.95],[.70,.60],[.75,.65],[.80,.70],[.85,.75],[.90,.80]])
X = np.vstack([normal, anomaly])
y = np.array([0] * len(normal) + [1] * len(anomaly))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=11, stratify=y
)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()


svc = SVC(kernel="rbf")
svc.fit(X_train, y_train)
pred_svm = svc.predict(X_test)
acc_svm = accuracy_score(y_test, pred_svm)

feature_map = zz_feature_map(feature_dimension=2, reps=2)
qkernel = FidelityStatevectorKernel(feature_map=feature_map)
qsvc = QSVC(quantum_kernel=qkernel)

qsvc.fit(X_train, y_train)
pred_qsvc = qsvc.predict(X_test)
acc_qsvc = accuracy_score(y_test, pred_qsvc)

print("Classical SVM Accuracy:", acc_svm)
print("Quantum SVM Accuracy :", acc_qsvc)

from sklearn.metrics import classification_report
print("학습/시험 크기:", X_train.shape, X_test.shape)
print("QSVM 분류 결과:")
print(classification_report(y_test, pred_qsvc, digits=3, zero_division=0))
K = qkernel.evaluate(X_train)
assert np.allclose(K, K.T) and np.allclose(np.diag(K), 1.0)
assert np.linalg.eigvalsh(K).min() > -1e-8

Classical SVM Accuracy: 1.0
Quantum SVM Accuracy : 0.875
학습/시험 크기: (16, 2) (8, 2)
QSVM 분류 결과:
              precision    recall  f1-score   support

           0      1.000     0.750     0.857         4
           1      0.800     1.000     0.889         4

    accuracy                          0.875         8
   macro avg      0.900     0.875     0.873         8
weighted avg      0.900     0.875     0.873         8



고정 입력 24건은 정상 12건과 이상 12건으로 구성되며 각 열은 0~1 범위의 무차원 특징 점수입니다. 첫 번째 열 범위는 0.10~0.90, 두 번째 열 범위는 0.12~0.95입니다. 라벨 0은 정상, 1은 이상입니다. 알려진 라벨로 학습하는 지도학습 실습입니다.

원자료를 학습 16건과 시험 8건으로 분할한 뒤 학습 자료로 정규화 기준을 구합니다. 2큐비트 zz_feature_map과 FidelityStatevectorKernel로 입력 상태 간 충실도를 계산하고 QSVC를 학습합니다. 논문 8.6절의 양자 특징 공간과 커널 개념을 교육용 데이터에 적용합니다. 논문의 실험 성능 재현 여부는 원래 데이터와 실험 조건을 적용하여 평가합니다.

실행 예에서 RBF SVM 정확도는 1.000, QSVM 정확도는 0.875입니다. QSVM은 시험 8건 중 7건을 맞혔으며 이상 클래스의 재현율은 1.000입니다. 상태벡터 계산은 이상적인 시뮬레이션이며 하드웨어 노이즈와 유한 측정 오차를 포함하지 않습니다. 실제 적용 성능은 데이터 규모, 분할, 인코딩과 커널 설정을 함께 바꾸어 평가합니다.